# Supplementary Methods 1-5: statistics

This notebook recomputes every number and statistical test printed in Supplementary Methods 1-5 (v28d lines 493-659) that this unit covers: the AlphaNovo permuted-prediction control, the bag-aware lookup comparison, the healthy-stool study-supported protein space, the parser audit, the CAMPI target databases and member presence, search input completeness, the human-assigned share with partly unannotated proteins, the annotation checks and the functional-allocation sensitivity. It reads only `source_data/`. The text has no figure panels, so there are no pixel checks.

Run from the repository root: `python paper.py build supplementary_methods_1_5`.

In [ ]:
from pathlib import Path
import sys
UNIT = Path.cwd()                 # the notebook runs with its own directory as cwd
REPO = UNIT.parents[1]
sys.path[:0] = [str(REPO), str(UNIT)]
SD = REPO / "source_data" / UNIT.name
OUT = REPO / "build" / UNIT.name   # gitignored; nothing is rendered by this unit
OUT.mkdir(parents=True, exist_ok=True)
import pandas as pd
pd.set_option("display.width", 200, "display.max_columns", 20)
import stats
print(stats.software())

## Statistic: AlphaNovo permuted-prediction control (line 553)

"Permuting predictions across spectra within each acquisition reduced the match rate to 0.006%" (v28d line 553). Unit: reference peptide-spectrum identification, n = 801,613 in 15 CAMPI acquisitions (9-50 residues). One permutation per acquisition was drawn upstream (`campi_alphanovo_accuracy`); this cell pools the per-acquisition counts. No test.

In [ ]:
stats.stat_alphanovo_permuted()

## Statistic: bag-aware and mass-equivalent lookup (lines 561, 579)

Gains and losses of peptide-acquisition identifications against exact lookup in 48 acquisitions (15 CAMPI, 32 healthy stool, 1 CapScan), seven arms each, nominal peptide q <= 0.01. Unit: acquisition. Counts are descriptive; no test.

In [ ]:
stats.stat_bags()

## Statistic: healthy-stool study-supported protein space (line 567)

"a study-supported protein space of 18,007,876 proteins" for the 58 acquisitions and "26,881,038" for the original 97-acquisition run (v28d line 567). Unit: reservoir protein. The 39 excluded acquisitions are the pooled prediction files that are not in the 58-acquisition roster.

In [ ]:
stats.stat_hstool_spaces()

## Statistic: parser audit (line 573)

"all 710 retained prediction files (112,835,982 rows)" (v28d line 573). Unit: prediction file. The v28f roster adds the 16 participant-15 CapScan files (726 files, 114,971,345 rows); the archived and current parsers agree on every file in both sets.

In [ ]:
stats.stat_parser_audit()

## Statistic: CAMPI target databases and member presence (lines 609, 611)

"29,673 reference and 19,435 MG records, each including 116 contaminants" (line 609) and "eight supplied member reference proteomes (29,557 member proteins)" (line 611). Unit: FASTA record and protein. The MG file holds 19,551 records because each of the 116 contaminants appears twice with identical name and sequence; 19,435 is the number of distinct records.

In [ ]:
display(stats.stat_campi_targets())
stats.stat_member_presence()

## Statistic: search input completeness (line 629)

"All passed (lowest coverage 99.6%)" (line 629). Unit: search (acquisition), n = 653 public-sequence searches (306 PREDICT, 347 CapScan; 637 in v28d) and 58 healthy-stool searches. Coverage is the highest scored scan divided by the declared spectrum count. The text states a 95% criterion; the original check used 90%. Both pass every search.

In [ ]:
stats.stat_completeness()

## Statistic: human-assigned share with partly unannotated compatible proteins (line 651)

"0.12% (healthy stool), 0.60% (CapScan) and 1.96% (PREDICT) of total LFQ intensity, pooled over all acquisitions" (line 651). Unit: LFQ intensity pooled over 58, 347 (v28f; 331 in v28d) and 306 acquisitions.

In [ ]:
stats.stat_partial_human()

## Statistic: annotation sources and the seed-ortholog check (line 657)

"Of 9,368,708 MGnify annotations, 372,277 (4.0%) failed this check, including 191,565 (2.0%) with no alignment"; "MGnify tables supplied annotations for 88-95% of compatible proteins"; "17,935 CAMPI reference proteins" (line 657). Unit: protein (gene annotation row). Failure means DIAMOND bitscore below half of the reported bitscore.

In [ ]:
stats.stat_annotations()

## Statistic: functional-allocation sensitivity, allocated share (line 657)

"These rules allocated a median of 7-61% instead of 62-81% of total intensity" (line 657). Unit: acquisition; medians per cohort, level and rule over 58 healthy-stool, 306 PREDICT and 346 non-empty CapScan acquisitions.

In [ ]:
med, ranges = stats.stat_allocated_fraction()
display(med)
ranges

## Statistic: functional-allocation sensitivity, CapScan own stool (line 657)

"All fourteen CapScan participants remained closer to their own stool (P≤1 × 10⁻⁵)" (line 657). Unit: participant, n = 15 on the v28f roster (14 in v28d). Two-sided permutation test, 99,999 random permutations of stool labels across participants (seed 20260910); P = 1 × 10⁻⁵ is the smallest value the test can return.

In [ ]:
stats.stat_own_stool()

## Statistic: functional-allocation sensitivity, PREDICT cross-omics (line 657)

"ρ=0.16-0.64 against 0.16-0.51 with equal allocation, all P≤0.026"; human-assigned layer "fell from 0.43-0.51 to 0.28-0.35 (P≤0.002)" (line 657). Unit: patient, n = 77. Mantel test with Spearman correlation of Euclidean distances, two-sided, 1,999 permutations (seed 20260907), no multiplicity correction.

In [ ]:
pm = stats.stat_predict_mantel()
display(pm)
stats.stat_predict_ranges(pm)

## Statistic: functional-allocation sensitivity, healthy-stool KO correlation (line 657)

"KO correlations still decreased from technical replicates (r=0.89-0.91) to different individuals, but the inter-individual value fell from 0.81 to 0.69-0.73" (line 657). Unit: acquisition pair (66 technical-replicate pairs, 45 inter-individual pairs); median of per-pair Pearson r of log2 KO intensity. Descriptive. The printed ranges pool the profile with and without human-assigned signal.

In [ ]:
hm, hr = stats.stat_hstool_r()
display(hm[hm.design.isin([stats.HSTOOL_TECHNICAL, stats.HSTOOL_INTER])])
hr

## Checks and all numbers

Compares the recomputed allocation-sensitivity results with the producers' own outputs, then recomputes every row of `NUMBERS.tsv` and `STATS.tsv` and prints the numbers in `NUMBERS.tsv` order. No panels, so no pixel checks.

In [ ]:
from tools.compare_values import same_numeric_value
print(stats.check_against_producers())
import csv
want = list(csv.DictReader(open(UNIT / "NUMBERS.tsv", encoding="utf-8"), delimiter="\t"))
got = stats.all_numbers()
assert len(want) == len(got)
for w, g in zip(want, got):
    assert {k: w[k] for k in stats.NUMBER_COLUMNS if k != "recomputed_value"} == {k: g[k] for k in stats.NUMBER_COLUMNS if k != "recomputed_value"}, (w, g)
    assert same_numeric_value(g["recomputed_value"], w["recomputed_value"]), (w, g)
    print(f"line {g['v28d_line']:>4}  printed {g['printed_value']:>12}  recomputed {g['recomputed_value']:<24} {g['status']}")
want_s = list(csv.DictReader(open(UNIT / "STATS.tsv", encoding="utf-8"), delimiter="\t"))
got_s = stats.all_stats()
assert len(want_s) == len(got_s)
for w, g in zip(want_s, got_s):
    # BLAS CPU kernels can change the last bits of Mantel rho. Only its
    # three result representations allow the existing 1e-12 relative tolerance.
    # Labels, P values, sample sizes, seeds and all other fields remain exact.
    from tools.compare_values import same_prefixed_numeric_value
    mantel_code = "text_statistics/supplementary_methods_1_5/stats.py:mantel"
    prefixes = {"estimate": "Mantel rho ", "statistic": "rho = ", "effect_size": ""}
    for k in stats.STAT_COLUMNS:
        if k == "software_version":
            continue
        if w["code"] == g["code"] == mantel_code and k in prefixes:
            assert same_prefixed_numeric_value(g[k], w[k], prefixes[k]), (w["claim"], k, w[k], g[k])
        else:
            assert w[k] == g[k], (w["claim"], k, w[k], g[k])
print(len(got), "numbers and", len(got_s), "statistics rows agree with NUMBERS.tsv and STATS.tsv")
print("pixel checks: none (text-only unit)")